# 01 · Data exploration
Structure, distributions and data quality of the standardised dataset.

> **Prerequisite:** `python -m gazegrounded.cli run-all` (synthetic demo).
>
> All logic lives in `src/gazegrounded/`; this notebook only calls it. With the synthetic config every number is **SYNTHETIC DEMO DATA — NOT RESEARCH EVIDENCE**.

In [1]:
from pathlib import Path
import os
# Run from the repository root so relative config paths resolve.
while not Path("pyproject.toml").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
import numpy as np
import pandas as pd
from gazegrounded.config import load_config
from gazegrounded.data.loader import load_processed
from gazegrounded.utils import SYNTHETIC_BANNER
cfg = load_config("configs/synthetic.yaml")  # switch to configs/experiment.yaml for real data
ds = load_processed(cfg)
print(SYNTHETIC_BANNER if ds.is_synthetic else "Real data:", cfg.data.source)

SYNTHETIC DEMO DATA — NOT RESEARCH EVIDENCE synthetic


## Tables
Every source is converted by an adapter into the same tables (see `data/schemas.py`).

In [2]:
for name in ('uis', 'aois', 'tasks', 'trials', 'gaze_samples'):
    df = getattr(ds, name)
    print(f'{name:13s} {len(df):7d} rows  columns: {list(df.columns)[:8]}')
ds.trials.head()

uis                 5 rows  columns: ['ui_id', 'image_path', 'width', 'height', 'domain', 'condition', 'pair_id']
aois               53 rows  columns: ['ui_id', 'aoi_id', 'label', 'x_min', 'y_min', 'x_max', 'y_max', 'semantic_role']
tasks              10 rows  columns: ['task_id', 'ui_id', 'task_key', 'task_description', 'target_aoi_ids', 'designed_friction_type']
trials            120 rows  columns: ['participant_id', 'session_id', 'trial_id', 'ui_id', 'task_id', 'task_description', 'condition', 'trial_order']
gaze_samples    33562 rows  columns: ['participant_id', 'session_id', 'trial_id', 'ui_id', 'timestamp_ms', 'gaze_x', 'gaze_y', 'validity']


,participant_id,session_id,trial_id,ui_id,task_id,task_description,condition,trial_order,task_success,completion_time_ms,error_count,click_count,feedback_text,feedback_category,friction_label,label_source,ease_rating
0,P01,P01_S1,P01_T01,shop_checkout_friction,shop_checkout_friction__change_address,Change the delivery address for this order.,friction,1,True,8101.0,2,3,Hard to find. I looked all over the page befor...,findability,high,synthetic_latent,3
1,P01,P01_S1,P01_T02,shop_checkout_clean,shop_checkout_clean__change_address,Change the delivery address for this order.,clean,2,True,1401.3,0,1,"No issues, I found it immediately.",no_friction,low,synthetic_latent,5
2,P01,P01_S1,P01_T03,travel_booking,travel_booking__find_baggage,Find out how much checked baggage is included.,single,3,True,1314.3,0,1,NaN,NaN,low,synthetic_latent,6
3,P01,P01_S1,P01_T04,shop_checkout_friction,shop_checkout_friction__apply_discount,Apply the discount code SAVE10 to your order.,friction,4,False,8774.0,4,4,The label didn't make it clear what it does.,ambiguous_wording,high,synthetic_latent,3
4,P01,P01_S1,P01_T05,bank_cards_friction,bank_cards_friction__find_limit,Find the card's monthly spending limit.,friction,5,True,8445.6,2,3,"Too much text on the page, the spending limit ...",information_overload,high,synthetic_latent,3


## Design and outcomes per condition

In [3]:
ds.trials.groupby('condition').agg(trials=('trial_id','count'), success=('task_success','mean'),
    completion_s=('completion_time_ms', lambda s: s.mean()/1000), errors=('error_count','mean'),
    ease=('ease_rating','mean')).round(2)

,trials,success,completion_s,errors,ease
condition,,,,,
clean,48,1.00,2.55,0.23,6.21
friction,48,0.81,7.99,1.67,2.77
single,24,0.96,4.48,0.83,4.04


In [4]:
pd.crosstab(ds.trials['condition'], ds.trials['friction_label'])

friction_label,high,low,medium
condition,,,
clean,0,43,5
friction,40,0,8
single,9,9,6


## Data quality
Valid-sample ratio per trial (blinks, track loss). Trials below the configured threshold are flagged, not silently dropped.

In [5]:
from gazegrounded.data.preprocessing import trial_quality
q = trial_quality(ds.gaze_samples, cfg.gaze.min_valid_sample_ratio)
print(q.groupby('participant_id')['valid_ratio'].mean().round(2).to_dict())
print('low-quality trials:', int(q['low_quality'].sum()))

{'P01': 0.95, 'P02': 0.93, 'P03': 0.92, 'P04': 0.91, 'P05': 0.95, 'P06': 0.9, 'P07': 0.54, 'P08': 0.94, 'P09': 0.91, 'P10': 0.93, 'P11': 0.93, 'P12': 0.94}
low-quality trials: 10


In [6]:
report = ds.validate()
print(report.summary()); report.warnings

0 errors, 1 warnings


['[gaze_samples] 10 trials have <60% valid samples (e.g. P06_T01)']